# M1-B1 — Modélisation

Tâches 4 à 8 : premier modèle, v1 rejouée, pipeline v2, validation et réglage, verdict.

Le chargement et le découpage viennent de `train.py`. Seule la tâche 8 lit le jeu de test.

In [7]:
import os
import sys
import tempfile
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, fbeta_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_validate, train_test_split
from sqlalchemy import create_engine, text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

# Avertissement sans effet (scikit-learn 1.6.1 avec un SciPy plus récent)
os.environ["PYTHONWARNINGS"] = "ignore:Unknown solver options"
warnings.filterwarnings("ignore", message="Unknown solver options")

from features import LEAKY_COLUMNS, NEW_CATEGORICAL, NEW_NUMERICAL, RAW_FEATURES, TARGET, USELESS_COLUMNS, add_features
from model.train_baseline import FEATURES as V1_FEATURES
from model.train_baseline import build_pipeline as build_v1
from train import CV, F2, HYPERPARAMETERS, build_pipeline, evaluate, load_data, split_train_test

engine = create_engine(os.environ["DATABASE_URL"])
df = load_data(engine)
train, test = split_train_test(df)
y_train = train[TARGET]

print(f"Entraînement : {len(train)} réservations, arrivées du {train['arrival_date'].min().date()} au {train['arrival_date'].max().date()}")
print(f"Taux d'annulation sur l'entraînement : {y_train.mean():.1%}")
print(f"Test : {len(test)} réservations, mises de côté jusqu'à la tâche 8")

Entraînement : 58792 réservations, arrivées du 2015-07-01 au 2016-08-31
Taux d'annulation sur l'entraînement : 35.6%
Test : 19911 réservations, mises de côté jusqu'à la tâche 8


## Tâche 4 — Premier modèle

Forêt aléatoire sur toutes les colonnes, encodées avec `pd.get_dummies`, dates exclues. Le score est mesuré sur 25 % du jeu d'entraînement.

In [8]:
def modele_naif(colonnes_retirees=()):
    """Forêt aléatoire : score de validation et 10 variables les plus importantes."""
    X = pd.get_dummies(train.drop(columns=[TARGET, "arrival_date", "reservation_status_date", *colonnes_retirees]))
    X_a, X_val, y_a, y_val = train_test_split(X, y_train, test_size=0.25, stratify=y_train, random_state=42)
    foret = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42).fit(X_a, y_a)
    print(f"{X.shape[1]} colonnes | accuracy de validation : {foret.score(X_val, y_val):.4f}")
    importances = pd.Series(foret.feature_importances_, index=X.columns).sort_values(ascending=False)
    return importances.head(10).round(3).to_frame("importance")


modele_naif()

233 colonnes | accuracy de validation : 1.0000


,importance
reservation_status_Canceled,0.328
reservation_status_Check-Out,0.315
deposit_type_Non Refund,0.079
previous_cancellations,0.040
deposit_type_No Deposit,0.039
country_PRT,0.025
lead_time,0.018
total_of_special_requests,0.014
reservation_status_No-Show,0.013
agent,0.011


L'accuracy est de 100 %. `reservation_status_Canceled` et `reservation_status_Check-Out` portent 64 % de l'importance.

### Contenu de `reservation_status`

Requête limitée à la période d'entraînement.

In [9]:
query = text("""
    SELECT reservation_status,
           is_canceled,
           COUNT(*)                                                AS nb_reservations,
           ROUND(AVG(reservation_status_date - arrival_date), 1)   AS jours_entre_arrivee_et_statut
    FROM reservations
    WHERE arrival_date <= :date_limite
    GROUP BY reservation_status, is_canceled
    ORDER BY reservation_status
""")
pd.read_sql(query, engine, params={"date_limite": train["arrival_date"].max().date()})

,reservation_status,is_canceled,nb_reservations,jours_entre_arrivee_et_statut
0,Canceled,1,20206,-75.4
1,Check-Out,0,37883,3.4
2,No-Show,1,703,-0.2


`reservation_status` reprend la cible : `Canceled` et `No-Show` correspondent à `is_canceled = 1`, `Check-Out` à 0. Ce statut est connu après le dénouement de la réservation.

`reservation_status_date` dépend aussi du dénouement : 75 jours avant l'arrivée en moyenne pour une annulation, 3 jours après pour un séjour honoré.

### Sans `reservation_status`

In [10]:
modele_naif(["reservation_status"])

230 colonnes | accuracy de validation : 0.9167


,importance
lead_time,0.082
country_PRT,0.077
deposit_type_Non Refund,0.076
reservation_id,0.069
deposit_type_No Deposit,0.063
previous_cancellations,0.058
adr,0.053
total_of_special_requests,0.049
agent,0.041
arrival_date_day_of_month,0.035


L'accuracy passe à 91,7 %. `reservation_id` est la 4e variable la plus importante. Les autres colonnes sont vérifiées ci-dessous.

In [11]:
# Lien entre reservation_id et l'ordre des arrivées
print("Corrélation de rang entre reservation_id et arrival_date :",
      round(train["reservation_id"].corr(train["arrival_date"].astype("int64"), method="spearman"), 4))

# Taux d'annulation selon cinq indices
indices = {
    "chambre attribuée différente de la chambre réservée": train["assigned_room_type"] != train["reserved_room_type"],
    "au moins une modification (booking_changes)": train["booking_changes"] > 0,
    "place de parking demandée": train["required_car_parking_spaces"] > 0,
    "au moins une demande particulière": train["total_of_special_requests"] > 0,
    "client du Portugal (country = PRT)": train["country"] == "PRT",
}
pd.DataFrame(
    {
        nom: {
            "nb_reservations": int(masque.sum()),
            "taux_annulation_si_oui": round(y_train[masque].mean(), 3),
            "taux_annulation_si_non": round(y_train[~masque].mean(), 3),
        }
        for nom, masque in indices.items()
    }
).T

Corrélation de rang entre reservation_id et arrival_date : 1.0


,nb_reservations,taux_annulation_si_oui,taux_annulation_si_non
chambre attribuée différente de la chambre réservée,8870.0,0.043,0.411
au moins une modification (booking_changes),8442.0,0.141,0.392
place de parking demandée,3966.0,0.000,0.381
au moins une demande particulière,20950.0,0.172,0.457
client du Portugal (country = PRT),28662.0,0.550,0.171


| Moment où l'information est connue | Colonnes | Décision |
|---|---|---|
| À la réservation | délai, dates d'arrivée, nuits, voyageurs, repas, segment, canal, historique du client, chambre réservée, acompte, agence, entreprise, type de client, prix | gardées |
| À la réservation, peut-être complétées ensuite | `country`, `required_car_parking_spaces`, `total_of_special_requests` | gardées, sous réserve |
| Pendant la vie de la réservation | `booking_changes` | retirée |
| À l'arrivée | `assigned_room_type` | retirée |
| Après le dénouement | `reservation_status`, `reservation_status_date` | retirées |
| Sans information utile | `reservation_id`, `hotel_id`, `arrival_date_year` | retirées |

- `assigned_room_type` : attribuée à l'arrivée. 4,3 % d'annulation si elle diffère de la chambre réservée, 41,1 % sinon.
- `booking_changes` : cumul jusqu'au dénouement. 14,1 % d'annulation avec au moins une modification, 39,2 % sans.
- `reservation_id` : corrélation de rang de 1 avec la date d'arrivée. `hotel_id` double `hotel`. `arrival_date_year` ne couvre pas les années futures.
- `country`, `required_car_parking_spaces`, `total_of_special_requests` : écarts forts (aucune annulation sur 3 966 réservations avec parking). Elles peuvent être complétées à l'arrivée. Elles sont gardées, car la v1 les utilise et l'API de M0-B1 les demande. Leur poids est mesuré en tâche 7.

In [12]:
modele_naif(["reservation_status", *[c for c in LEAKY_COLUMNS + USELESS_COLUMNS if not c.endswith("_date")]])

214 colonnes | accuracy de validation : 0.9109


,importance
lead_time,0.100
deposit_type_No Deposit,0.082
previous_cancellations,0.072
country_PRT,0.071
deposit_type_Non Refund,0.069
adr,0.069
total_of_special_requests,0.053
arrival_date_day_of_month,0.046
agent,0.045
arrival_date_week_number,0.044


Après retrait, l'accuracy est de 91,1 %. La validation est tirée au hasard dans la même période. La tâche 5 utilise une validation croisée temporelle.

## Tâche 5 — v1 rejouée

La recette de la v1 (`build_pipeline()` de `model/train_baseline.py`) est entraînée sur le jeu d'entraînement et mesurée par validation croisée.

- Score suivi : le F2 de la classe « annulée », où le rappel pèse deux fois plus que la précision. Le choix est justifié en tâche 8.
- Validation : 5 folds classiques, puis `TimeSeriesSplit`, qui valide sur une période postérieure à l'entraînement. Les données sont triées par date d'arrivée.

In [13]:
SCORING = {"f2": F2, "recall": "recall", "precision": "precision", "roc_auc": "roc_auc"}
bilan = {}


def valider(nom, pipeline, X, cv=CV):
    """Validation croisée sur l'entraînement : ajoute une ligne au bilan, renvoie les scores par fold."""
    scores = pd.DataFrame(
        cross_validate(pipeline, X, y_train, cv=cv, scoring=SCORING, return_train_score=True, n_jobs=-1)
    )
    bilan[nom] = {
        "F2 validation": scores["test_f2"].mean(),
        "écart-type": scores["test_f2"].std(),
        "F2 entraînement": scores["train_f2"].mean(),
        "écart entr. - val.": scores["train_f2"].mean() - scores["test_f2"].mean(),
        "rappel": scores["test_recall"].mean(),
        "précision": scores["test_precision"].mean(),
        "ROC AUC": scores["test_roc_auc"].mean(),
    }
    return scores


def tableau(*noms):
    return pd.DataFrame(bilan).T.loc[list(noms)].round(3)


valider("v1 rejouée (5 folds classiques)", build_v1(), train[V1_FEATURES], cv=5)
scores_v1 = valider("v1 rejouée", build_v1(), train[V1_FEATURES])
tableau("v1 rejouée (5 folds classiques)", "v1 rejouée")

,F2 validation,écart-type,F2 entraînement,écart entr. - val.,rappel,précision,ROC AUC
v1 rejouée (5 folds classiques),0.681,0.145,0.860,0.179,0.678,0.782,0.877
v1 rejouée,0.644,0.194,0.912,0.268,0.613,0.853,0.891


In [14]:
# Détail par fold de la validation temporelle
periodes = [
    f"{train['arrival_date'].iloc[val].min().date()} → {train['arrival_date'].iloc[val].max().date()}"
    for _, val in CV.split(train)
]
detail = scores_v1[["test_f2", "test_recall", "test_precision", "train_f2"]].round(3)
detail.insert(0, "période de validation", periodes)
detail.insert(1, "taille entraînement", [len(tr) for tr, _ in CV.split(train)])
detail

,période de validation,taille entraînement,test_f2,test_recall,test_precision,train_f2
0,2015-09-19 → 2015-12-05,9802,0.856,0.846,0.899,0.962
1,2015-12-05 → 2016-03-10,19600,0.640,0.597,0.899,0.943
2,2016-03-10 → 2016-05-05,29398,0.640,0.603,0.846,0.904
3,2016-05-05 → 2016-06-30,39196,0.749,0.725,0.863,0.880
4,2016-06-30 → 2016-08-31,48994,0.335,0.294,0.757,0.872


Référence : F2 de 0,644 en validation croisée temporelle (rappel 0,613, précision 0,853).

- Écart entre entraînement et validation : 0,912 contre 0,644.
- Écart-type entre folds : 0,194. Sur le dernier fold (juillet et août 2016), le rappel est de 0,294.
- Les 5 folds classiques donnent 0,681. Ils s'entraînent aussi sur des réservations postérieures à celles qu'ils valident. La validation temporelle est utilisée pour la suite.